In [1]:
from model import agent, google_model
llm_agent = agent(google_model)

In [2]:
from langchain_core.messages import HumanMessage
result = llm_agent.invoke({"messages": [HumanMessage("Olá, tudo bem?")]} )
print(result["messages"][-1].content)


Olá! Tudo bem por aqui. Em que posso ajudar?


In [3]:
msg = HumanMessage(
    content=(
        "Carregue o arquivo 'rba-dataset.csv'. "
        "Analise sua estrutura e informe: "
        "1) Quantas colunas existem; "
        "2) Os nomes de cada coluna; "
        "3) O tipo de dado predominante em cada uma. "
        "Se possível, identifique colunas que pareçam ser IDs, datas, textos ou categorias."
    )
)
result = llm_agent.invoke({"messages": [msg]})
print(result["messages"][-1].content)

O arquivo 'rba-dataset.csv' possui 16 colunas.

**Nomes das colunas:**
- index
- Login Timestamp
- User ID
- Round-Trip Time [ms]
- IP Address
- Country
- Region
- City
- ASN
- User Agent String
- Browser Name and Version
- OS Name and Version
- Device Type
- Login Successful
- Is Attack IP
- Is Account Takeover

**Tipos de dados predominantes e identificação:**
- **index**: Inteiro (ID numérico sequencial)
- **Login Timestamp**: Texto/Timestamp (Data e hora)
- **User ID**: Numérico (ID de usuário, parece ser categórico/nominal com valores muito grandes)
- **Round-Trip Time [ms]**: Numérico (Tempo em milissegundos, pode conter valores ausentes - NaN)
- **IP Address**: Texto (Endereços IP)
- **Country**: Texto (Códigos de país - Categoria)
- **Region**: Texto (Regiões geográficas - Categoria)
- **City**: Texto (Cidades - Categoria)
- **ASN**: Inteiro (Sistema Autônomo de Rede - Categoria/ID numérico)
- **User Agent String**: Texto (Informações detalhadas do navegador/SO/dispositivo)
- *

In [7]:
from langchain_core.messages import HumanMessage, SystemMessage
messages = [
    SystemMessage(content=
        "Você é um analista de segurança da informação especializado em logs de autenticação "
        "e correlação com MITRE ATT&CK. Seja direto, objetivo e baseado exclusivamente no dataset."
    ),

    HumanMessage(content=
        "Carregue o arquivo 'rba-dataset.csv'. "
        "A seguir, execute SOMENTE as análises abaixo:"
    ),

    HumanMessage(content=
        "1) Conte quantos logins falharam no dataset (Login Successful == False). "
        "2) Conte quantos logins foram bem-sucedidos. "
        "3) Conte quantos eventos vieram de IPs marcados como 'Is Attack IP == True'. "
        "4) Conte quantos eventos foram marcados como 'Is Account Takeover == True'. "
        "5) Identifique usuários com mais de 3 falhas consecutivas (se possível). "
        "6) Identifique padrões suspeitos (ex.: muitas falhas seguidas de sucesso). "
        "7) Para cada achado, forneça o mapeamento MITRE ATT&CK (TAxxxx + Txxxx) em 1 frase."
    ),

    HumanMessage(content=
        "Formato da resposta:\n"
        "• Quantidades (valores exatos)\n"
        "• Relações encontradas (bullets curtos)\n"
        "• MITRE (por item)\n"
        "Não liste colunas. Não forneça explicações longas."
    )
]


result = llm_agent.invoke({"messages": messages })
print(result["messages"][-1].content)

*   **Logins Falharam:** 10
*   **Logins Bem-sucedidos:** 10
*   **Eventos de IPs de Ataque:** 10
*   **Eventos de Tomada de Conta:** 10

*   **Anomalias:**
    *   10 eventos de logins falharam. MITRE ATT&CK: T1110 - Brute force de senhas.
    *   10 eventos de logins bem-sucedidos. MITRE ATT&CK: T1078 - Contas válidas usadas para acesso inicial.
    *   10 eventos de IPs de ataque. MITRE ATT&CK: T1071 - Comunicação de Aplicação.
    *   10 eventos de tomada de conta. MITRE ATT&CK: T1078 - Contas válidas usadas para acesso inicial.

*   **Padrões Suspeitos:**
    *   Combinação de logins falhos e bem-sucedidos. MITRE ATT&CK: T1110 - Brute force de senhas, seguido por T1078 - Contas válidas usadas para acesso inicial.
    *   Logins de IPs de ataque que resultaram em sucesso ou falha. MITRE ATT&CK: T1071 - Comunicação de Aplicação, em conjunto com T1110 ou T1078.
    *   Tomada de conta a partir de IPs de ataque. MITRE ATT&CK: T1078 - Contas válidas usadas para acesso inicial, com orig

In [8]:
messages = [
    SystemMessage(content=
        "Você é um analista de segurança especializado em logs de autenticação e MITRE ATT&CK. "
        "Responda apenas com dados objetivos do dataset, sem explicações longas."
    ),

    HumanMessage(content=
        "Carregue o arquivo 'rba-dataset.csv' e realize SOMENTE as análises abaixo:"
    ),

    HumanMessage(content=
        "1) Contar quantos logins falharam (Login Successful == False). "
        "   • Listar os User ID envolvidos nas falhas (sem duplicar). "
        "2) Contar quantos logins foram bem-sucedidos. "
        "   • Listar os User ID desses logins. "
        "3) Contar quantos eventos vieram de IPs 'Is Attack IP == True'. "
        "   • Listar os User ID desses eventos. "
        "4) Contar quantos eventos foram marcados como 'Is Account Takeover == True'. "
        "   • Listar os User ID desses eventos. "
        "5) Identificar usuários com mais de 3 falhas consecutivas. "
        "   • Retornar somente os User ID identificados. "
        "6) Identificar padrões suspeitos como: falhas seguidas de sucesso, logins de locais incomuns ou mudanças bruscas de device/OS. "
        "   • Listar os User ID detectados para cada padrão. "
        "7) Para cada achado, retorne um único mapeamento MITRE ATT&CK (TAxxxx + Txxxx) em 1 frase."
    ),

    HumanMessage(content=
        "Formato da resposta:\n"
        "• Quantidades (valores exatos)\n"
        "• User ID listados por categoria\n"
        "• MITRE por item (1 linha)\n"
        "Nada de listar colunas, nada de parágrafos."
    )
]
result = llm_agent.invoke({"messages": messages })
print(result["messages"][-1].content)

1) Contagem de logins falhados: 10
   User ID envolvidos: -4.3244755833065917e+18, 5.72967953528197e+18, -4.6188540719426212e+18, -3.2439787248024351e+18, 8.0760005525873695e+18
2) Contagem de logins bem-sucedidos: 10
   User ID envolvidos: 5.9325019382874122e+18, -9.08082924386383e+18, -8.2966672062737644e+18, -4.6639435259438612e+18, 1.2112990189800197e+18, -3.2841374792624333e+18, 6.3866375502237676e+18, -6.8242061793197158e+18, 7.24653344389824e+18, -3.0659361405498563e+18
3) Contagem de eventos de IPs de ataque: 10
   User ID envolvidos: -4.3244755833065917e+18, 9.13347065560009e+18, 9.13347065560009e+18, 9.13347065560009e+18, -4.6188540719426212e+18, -4.3244755833065917e+18, -3.522288621475838e+18, -6.6232180665668178e+18, -4.3244755833065917e+18, 4.9449806330135104e+17
4) Contagem de eventos de "Account Takeover": 10
   User ID envolvidos: -6.3802560631651461e+18, 4.13007443916652e+18, -1.3695593091789229e+17, -5.7838010280788756e+18, 6.9694918051670282e+18, -2.2004911887124631e

In [63]:
from langchain_core.messages import HumanMessage, SystemMessage
messages = [
    SystemMessage(content="Você é um analista de dados especialista em ITSM."),
    HumanMessage(content="Explique o dataset."),
    HumanMessage(content="Agora conte os incidentes por prioridade."),
    HumanMessage(content="Agora gere insights avançados.")
]
result = llm_agent.invoke({"messages": messages })
print(result["messages"][-1].content)

O dataset contém 141.712 incidentes de TI. Cada registro inclui detalhes como o estado do incidente, contagem de reatribuições, informações do chamador, localização, categoria, impacto, urgência, prioridade, grupo atribuído, e timestamps de criação e atualização.

A contagem de incidentes por prioridade é a seguinte:
- 3 - Moderate: 141708 incidentes

**Insights Avançados:**

1.  **Dominância de Prioridade Média:** A esmagadora maioria dos incidentes (141.708 de 141.712) são classificados como '3 - Moderate' (Média). Isso sugere que ou a maioria dos problemas de TI são de impacto e urgência moderados, ou que há uma tendência a classificar os incidentes com essa prioridade. Seria interessante investigar se incidentes com prioridade 'Alta' ou 'Crítica' são subnotificados ou se os critérios de classificação precisam ser revisados.

2.  **Análise de SLA:** A coluna `made_sla` indica se o incidente foi resolvido dentro do Acordo de Nível de Serviço (SLA). Com 141.708 incidentes de prioridad

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
messages = [
    SystemMessage(content=
        "Você é um analista de segurança da informação especializado em logs de autenticação "
        "e correlação com MITRE ATT&CK. Seja direto, objetivo e baseado exclusivamente no dataset."
    ),

    HumanMessage(content=
        "Carregue o arquivo 'rba-dataset.csv'. "
        "A seguir, execute SOMENTE as análises abaixo:"
    ),

    HumanMessage(content=
        "1) Conte quantos logins falharam no dataset (Login Successful == False). "
        "2) Conte quantos logins foram bem-sucedidos. "
        "3) Conte quantos eventos vieram de IPs marcados como 'Is Attack IP == True'. "
        "4) Conte quantos eventos foram marcados como 'Is Account Takeover == True'. "
        "5) Identifique usuários com mais de 3 falhas consecutivas (se possível). "
        "6) Identifique padrões suspeitos (ex.: muitas falhas seguidas de sucesso). "
        "7) Para cada achado, forneça o mapeamento MITRE ATT&CK (TAxxxx + Txxxx) em 1 frase."
    ),

    HumanMessage(content=
        "Formato da resposta:\n"
        "• Quantidades (valores exatos)\n"
        "• Relações encontradas (bullets curtos)\n"
        "• MITRE (por item)\n"
        "Não liste colunas. Não forneça explicações longas."
    )
]


result = llm_agent.invoke({"messages": messages })
print(result["messages"][-1].content)